---

# Section 0: Configuration Globale et Imports

---


## 0.1 Imports des Bibliothèques


In [30]:
# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import json
import os
import time
from collections import Counter
# Imports scipy pour matrices sparse
from scipy.sparse import csr_matrix, coo_matrix
# Imports LightFM
try:
    from lightfm import LightFM
    from lightfm.data import Dataset
    from lightfm.evaluation import precision_at_k, recall_at_k, auc_score
    from lightfm.cross_validation import random_train_test_split
    print("✅ LightFM installé et disponible")
    LIGHTFM_AVAILABLE = True
except ImportError:
    print("❌ LightFM n'est pas installé!")
    print("   Installer avec: pip install lightfm")
    LIGHTFM_AVAILABLE = False
    raise ImportError("LightFM est requis pour ce notebook")
# Configuration des warnings et affichage
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
# Style des visualisations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
print("\n✅ Imports terminés")


✅ LightFM installé et disponible

✅ Imports terminés


## 0.2 Configuration Globale du Projet


In [31]:
# ============================================================================
# PARAMÈTRES GLOBAUX - À MODIFIER SELON VOS BESOINS
# ============================================================================

# Taille du sample (nombre de transactions à échantillonner)
SAMPLE_SIZE = '50K'  # Options: 1000, 10000, 50000

# Stratégie de sampling
MIN_USER_TRANSACTIONS = 5   # Users avec au moins N transactions (dans dataset complet)
MIN_ITEM_TRANSACTIONS = 10  # Items avec au moins N transactions (dans dataset complet)

# Features sélectionnées
ITEM_FEATURE_COLUMNS = [
    'product_group_name',
    'product_type_name',
    'garment_group_name',
    'colour_group_name',
    'department_name'
]


USER_FEATURE_COLUMNS = [
    'age_group',
    'club_member_status',
    'fashion_news_frequency'
]

# Paramètres de split
TEMPORAL_TRAIN_RATIO = 0.8  # 80% train, 20% test pour split temporel
RANDOM_TEST_PERCENTAGE = 0.2  # 20% test pour split aléatoire
USERBASED_TRAIN_RATIO = 0.8  # 80% users train, 20% users test

# Choix de la stratégie de split pour l'entraînement final
SPLIT_STRATEGY = 'temporal'  # Options: 'temporal', 'random', 'userbased'

# Paramètres d'entraînement
N_EPOCHS = 10
N_THREADS = 4

# Paramètres de grid search
GRID_SEARCH_PARAMS = {
    'no_components': [10, 30, 50, 100],
    'learning_rate': [0.01, 0.05, 0.1],
    'item_alpha': [1e-6, 1e-5, 1e-4],
    'user_alpha': [1e-6, 1e-5, 1e-4]
}

# Métriques d'évaluation
K_VALUES = [5, 10, 20]  # Pour Precision@K, Recall@K

# Seed pour reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Chemins des données
DATA_PATH = 'data/'
TRANSACTIONS_FILE = DATA_PATH + 'transactions_train.csv'
ARTICLES_FILE = DATA_PATH + 'articles.csv'
CUSTOMERS_FILE = DATA_PATH + 'customers.csv'

# ============================================================================
# AFFICHAGE DE LA CONFIGURATION
# ============================================================================
print("="*80)
print("CONFIGURATION DU PROJET")
print("="*80)
print(f"\n📊 Dataset:")
print(f"   • Taille du sample: {SAMPLE_SIZE} transactions")
print(f"   • Min transactions/user (filtrage): {MIN_USER_TRANSACTIONS}")
print(f"   • Min transactions/item (filtrage): {MIN_ITEM_TRANSACTIONS}")
print(f"\n🎯 Features:")
print(f"   • Item features: {len(ITEM_FEATURE_COLUMNS)} colonnes")
print(f"   • User features: {len(USER_FEATURE_COLUMNS)} colonnes")
print(f"\n🔀 Split Strategy:")
print(f"   • Stratégie choisie: {SPLIT_STRATEGY.upper()}")
print(f"   • Temporal ratio: {TEMPORAL_TRAIN_RATIO*100:.0f}% train")
print(f"   • Random test: {RANDOM_TEST_PERCENTAGE*100:.0f}%")
print(f"\n🤖 Entraînement:")
print(f"   • Epochs: {N_EPOCHS}")
print(f"   • Threads: {N_THREADS}")
print(f"   • Random state: {RANDOM_STATE}")
print(f"\n🔍 Grid Search:")
print(f"   • Paramètres à tester:")
for param, values in GRID_SEARCH_PARAMS.items():
    print(f"     - {param}: {values}")
total_combinations = np.prod([len(v) for v in GRID_SEARCH_PARAMS.values()])
print(f"   • Total combinaisons: {total_combinations}")
print(f"\n✅ Configuration chargée")
print("="*80)


CONFIGURATION DU PROJET

📊 Dataset:
   • Taille du sample: 50K transactions
   • Min transactions/user (filtrage): 5
   • Min transactions/item (filtrage): 10

🎯 Features:
   • Item features: 5 colonnes
   • User features: 3 colonnes

🔀 Split Strategy:
   • Stratégie choisie: TEMPORAL
   • Temporal ratio: 80% train
   • Random test: 20%

🤖 Entraînement:
   • Epochs: 10
   • Threads: 4
   • Random state: 42

🔍 Grid Search:
   • Paramètres à tester:
     - no_components: [10, 30, 50, 100]
     - learning_rate: [0.01, 0.05, 0.1]
     - item_alpha: [1e-06, 1e-05, 0.0001]
     - user_alpha: [1e-06, 1e-05, 0.0001]
   • Total combinaisons: 108

✅ Configuration chargée


## 0.3 Fonctions Utilitaires


In [32]:
def print_section_header(title, section_number=None):
    """Affiche un header de section formaté."""
    print("\n" + "="*80)
    if section_number:
        print(f"SECTION {section_number}: {title.upper()}")
    else:
        print(title.upper())
    print("="*80 + "\n")

def print_subsection_header(title):
    """Affiche un header de sous-section formaté."""
    print("\n" + "-"*80)
    print(title)
    print("-"*80 + "\n")

def print_dataframe_info(df, name):
    """Affiche des informations sur un DataFrame."""
    print(f"\n📊 {name}:")
    print(f"   • Shape: {df.shape}")
    print(f"   • Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"   • Colonnes: {list(df.columns)}")

def calculate_sparsity(n_interactions, n_users, n_items):
    """Calcule la sparsité d'une matrice user-item."""
    return 1 - (n_interactions / (n_users * n_items))

def format_large_number(num):
    """Formate un grand nombre avec des séparateurs."""
    return f"{num:,}"

def timer(func):
    """Décorateur pour mesurer le temps d'exécution."""
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"\n⏱️  Temps d'exécution: {end-start:.2f} secondes")
        return result
    return wrapper

def evaluate_model(model, test_interactions, train_interactions=None, 
                   item_features=None, user_features=None, k=10):
    """Évalue un modèle LightFM avec plusieurs métriques."""
    metrics = {}
    # Precision@K
    precision = precision_at_k(
        model, test_interactions, 
        train_interactions=train_interactions,
        item_features=item_features,
        user_features=user_features,
        k=k
    ).mean()
    metrics[f'precision@{k}'] = precision
    # Recall@K
    recall = recall_at_k(
        model, test_interactions,
        train_interactions=train_interactions,
        item_features=item_features,
        user_features=user_features,
        k=k
    ).mean()
    metrics[f'recall@{k}'] = recall
    # AUC
    auc = auc_score(
        model, test_interactions,
        train_interactions=train_interactions,
        item_features=item_features,
        user_features=user_features
    ).mean()
    metrics['auc'] = auc
    return metrics
print("✅ Fonctions utilitaires chargées")


✅ Fonctions utilitaires chargées


---

# Section 3: Prétraitement et Construction LightFM

---


In [41]:
# Chemins des fichiers
DATA_PATH = 'data/'
print("📂 Chargement des données...")
print("-" * 50)
# Chargement des transactions
print("Chargement de transactions_train.csv...")
transactions = pd.read_csv(DATA_PATH + 'transactions_train.csv')
print(f"  ✓ {len(transactions):,} transactions chargées")
# Chargement des articles
print("\nChargement de articles.csv...")
articles = pd.read_csv(DATA_PATH + 'articles.csv')
print(f"  ✓ {len(articles):,} articles chargés")
# Chargement des clients
print("\nChargement de customers.csv...")
customers = pd.read_csv(DATA_PATH + 'customers.csv')
print(f"  ✓ {len(customers):,} clients chargés")
print("\n✅ Toutes les données ont été chargées avec succès !")

transactions_clean = transactions.drop_duplicates()
transactions_clean['t_dat'] = pd.to_datetime(transactions_clean['t_dat'])
print("\n✅ Toutes les données ont été nettoyées et transformés avec succès !")

📂 Chargement des données...
--------------------------------------------------
Chargement de transactions_train.csv...
  ✓ 31,788,324 transactions chargées

Chargement de articles.csv...
  ✓ 105,542 articles chargés

Chargement de customers.csv...
  ✓ 1,371,980 clients chargés

✅ Toutes les données ont été chargées avec succès !

✅ Toutes les données ont été nettoyées et transformés avec succès !


In [42]:
print("=" * 80)
print("CRÉATION DU DATASET PRÉ-FILTRÉ (STRATÉGIE COMBINÉE)")
print("=" * 80)

# Identifier les users actifs (≥5 transactions)
print("\n1️⃣  Identification des users actifs...")
user_activity = transactions_clean['customer_id'].value_counts()
users_actifs = user_activity[user_activity >= 5].index
print(f"   ✓ {len(users_actifs):,} users actifs (≥5 transactions)")
print(f"   • {len(user_activity[user_activity < 5]):,} users exclus (< 5 transactions)")

# Identifier les items populaires (≥10 transactions)
print("\n2️⃣  Identification des items populaires...")
item_activity = transactions_clean['article_id'].value_counts()
items_populaires = item_activity[item_activity >= 10].index
print(f"   ✓ {len(items_populaires):,} items populaires (≥10 transactions)")
print(f"   • {len(item_activity[item_activity < 10]):,} items exclus (< 10 transactions)")

# Filtrer le dataset complet pour créer base_sample_combine
print("\n3️⃣  Création du dataset pré-filtré 'Combiné'...")
base_sample_combine = transactions_clean[
    (transactions_clean['customer_id'].isin(users_actifs)) &
    (transactions_clean['article_id'].isin(items_populaires))
].copy()
print(f"   ✓ Dataset pré-filtré créé !")

CRÉATION DU DATASET PRÉ-FILTRÉ (STRATÉGIE COMBINÉE)

1️⃣  Identification des users actifs...
   ✓ 899,288 users actifs (≥5 transactions)
   • 462,993 users exclus (< 5 transactions)

2️⃣  Identification des items populaires...
   ✓ 82,396 items populaires (≥10 transactions)
   • 22,151 items exclus (< 10 transactions)

3️⃣  Création du dataset pré-filtré 'Combiné'...
   ✓ Dataset pré-filtré créé !


In [43]:
transactions = base_sample_combine.sample(n=10000, random_state=42).copy()

In [55]:
print("=" * 80)
print("CRÉATION DU DATASET LIGHTFM ET MAPPINGS")
print("=" * 80)

# Créer l'objet Dataset
dataset = Dataset()

# Récupérer les IDs uniques
unique_users = transactions['customer_id'].unique()
unique_items = transactions['article_id'].unique()
print(f"\n📊 Nombre d'IDs uniques à mapper:")
print(f"   Utilisateurs: {len(unique_users):,}")
print(f"   Articles: {len(unique_items):,}")

# Préparer les features pour le fit
print(f"\n🔄 Préparation des features pour fit()...")
# Item features : récupérer toutes les valeurs uniques
item_feature_columns = [
    'product_group_name',
    'product_type_name',
    'garment_group_name',
    'colour_group_name',
    'department_name'
]

# Filtrer articles pour le sample
articles_filtered = articles[articles['article_id'].isin(unique_items)].copy()

# Imputer valeurs manquantes AVANT de récupérer les features
for col in item_feature_columns:
    articles_filtered[col].fillna('Unknown', inplace=True)

# Récupérer toutes les features items uniques
all_item_features = set()
for col in item_feature_columns:
    unique_values = articles_filtered[col].unique()
    for val in unique_values:
        all_item_features.add(f"{col}:{val}")
print(f"   ✓ Item features uniques: {len(all_item_features):,}")
print(f"     Exemple: {list(all_item_features)[:5]}")

# User features : préparer les features users
customers_filtered = customers[customers['customer_id'].isin(unique_users)].copy()

# Imputer age et créer age_group
median_age = customers_filtered['age'].median()
customers_filtered['age'].fillna(median_age, inplace=True)
customers_filtered['age_group'] = pd.cut(
    customers_filtered['age'],
    bins=[0, 25, 35, 45, 55, 100],
    labels=['<25', '25-35', '35-45', '45-55', '55+'],
    include_lowest=True
).astype(str)

# Imputer autres features
customers_filtered['club_member_status'].fillna('ACTIVE', inplace=True)
customers_filtered['fashion_news_frequency'].fillna('NONE', inplace=True)

# Récupérer toutes les features users uniques
user_feature_columns = [
    'age_group',
    'club_member_status',
    'fashion_news_frequency'
]
all_user_features = set()
for col in user_feature_columns:
    unique_values = customers_filtered[col].unique()
    for val in unique_values:
        all_user_features.add(f"{col}:{val}")
print(f"   ✓ User features uniques: {len(all_user_features):,}")
print(f"     Exemple: {list(all_user_features)[:5]}")

# FIT du Dataset avec users, items, et features
print(f"\n🔄 Fitting du Dataset LightFM...")
print(f"   Cette étape crée les mappings internes...")
dataset.fit(
    users=unique_users,
    items=unique_items
)
print(f"   ✓ Dataset fitted avec succès!")

# Vérifier les dimensions
num_users, num_items = dataset.interactions_shape()
print(f"\n📊 Dimensions du Dataset:")
print(f"   Users: {num_users:,}")
print(f"   Items: {num_items:,}")
print(f"   Item features: {len(all_item_features):,}")
print(f"   User features: {len(all_user_features):,}")

# Créer des mappings inverses pour référence (optionnel, pour notre usage)
# Ces mappings nous permettent de récupérer les IDs originaux
user_id_mapping, user_features_mapping, item_id_mapping, item_features_mapping = dataset.mapping()
print(f"\n✅ Mappings créés avec succès")
print(f"   user_id_mapping: {len(user_id_mapping):,} entrées")
print(f"   item_id_mapping: {len(item_id_mapping):,} entrées")


CRÉATION DU DATASET LIGHTFM ET MAPPINGS

📊 Nombre d'IDs uniques à mapper:
   Utilisateurs: 9,851
   Articles: 7,865

🔄 Préparation des features pour fit()...
   ✓ Item features uniques: 387
     Exemple: ['department_name:Limited Edition', 'department_name:Expressive Lingerie', 'department_name:Dresses', 'garment_group_name:Woven/Jersey/Knitted mix Baby', 'product_group_name:Accessories']
   ✓ User features uniques: 11
     Exemple: ['fashion_news_frequency:Regularly', 'fashion_news_frequency:NONE', 'club_member_status:LEFT CLUB', 'age_group:55+', 'age_group:45-55']

🔄 Fitting du Dataset LightFM...
   Cette étape crée les mappings internes...
   ✓ Dataset fitted avec succès!

📊 Dimensions du Dataset:
   Users: 9,851
   Items: 7,865
   Item features: 387
   User features: 11

✅ Mappings créés avec succès
   user_id_mapping: 9,851 entrées
   item_id_mapping: 7,865 entrées


In [61]:
print("=" * 80)
print("STRATÉGIE 1: SPLIT TEMPOREL")
print("=" * 80)

# Configuration
TRAIN_RATIO = TEMPORAL_TRAIN_RATIO
print(f"\n⚙️  Configuration:")
print(f"   Train: {TRAIN_RATIO*100:.0f}% (les plus anciennes)")
print(f"   Test: {(1-TRAIN_RATIO)*100:.0f}% (les plus récentes)")

# Trier par date
transactions_sorted = transactions.sort_values('t_dat').reset_index(drop=True)

# Calculer cutoff
cutoff_idx = int(len(transactions_sorted) * TRAIN_RATIO)
cutoff_date = transactions_sorted.iloc[cutoff_idx]['t_dat']
print(f"\n📅 Date de cutoff: {cutoff_date.date()}")

# Split
temporal_train_data = transactions_sorted[transactions_sorted['t_dat'] < cutoff_date].copy()
temporal_test_data = transactions_sorted[transactions_sorted['t_dat'] >= cutoff_date].copy()
print(f"\n📊 Split initial:")
print(f"   Train: {len(temporal_train_data):,} transactions ({len(temporal_train_data)/len(transactions)*100:.1f}%)")
print(f"   Test: {len(temporal_test_data):,} transactions ({len(temporal_test_data)/len(transactions)*100:.1f}%)")
print(f"\n   Train période: {temporal_train_data['t_dat'].min().date()} → {temporal_train_data['t_dat'].max().date()}")
print(f"   Test période: {temporal_test_data['t_dat'].min().date()} → {temporal_test_data['t_dat'].max().date()}")

# Filtrer pour garder seulement users/items communs
train_users = set(temporal_train_data['customer_id'])
train_items = set(temporal_train_data['article_id'])
test_users = set(temporal_test_data['customer_id'])
test_items = set(temporal_test_data['article_id'])
common_users = train_users & test_users
common_items = train_items & test_items
print(f"\n🔍 Analyse cold-start:")
print(f"   Users communs: {len(common_users):,} / {len(test_users):,} ({len(common_users)/len(test_users)*100:.1f}%)")
print(f"   Items communs: {len(common_items):,} / {len(test_items):,} ({len(common_items)/len(test_items)*100:.1f}%)")

# Filtrer test
temporal_test_data_filtered = temporal_test_data
#temporal_test_data_filtered = temporal_test_data[
#    (temporal_test_data['customer_id'].isin(common_users)) &
#    (temporal_test_data['article_id'].isin(common_items))
#].copy()
print(f"\n📊 Après filtrage:")
print(f"   Train: {len(temporal_train_data):,} transactions")
print(f"   Test: {len(temporal_test_data_filtered):,} transactions")
print(f"   Perte: {len(temporal_test_data) - len(temporal_test_data_filtered):,} transactions")

# Construire matrices avec LightFM
print(f"\n🔄 Construction des matrices LightFM...")
(temporal_train_interactions, _) = dataset.build_interactions(
    ((row['customer_id'], row['article_id']) 
     for idx, row in temporal_train_data.iterrows())
)
(temporal_test_interactions, _) = dataset.build_interactions(
    ((row['customer_id'], row['article_id']) 
     for idx, row in temporal_test_data_filtered.iterrows())
)
print(f"   ✓ Train: {temporal_train_interactions.shape}, {temporal_train_interactions.nnz:,} nnz")
print(f"   ✓ Test: {temporal_test_interactions.shape}, {temporal_test_interactions.nnz:,} nnz")

# Stats
temporal_sparsity_train = 1 - (temporal_train_interactions.nnz / (num_users * num_items))
temporal_sparsity_test = 1 - (temporal_test_interactions.nnz / (num_users * num_items))
print(f"\n📈 Sparsité:")
print(f"   Train: {temporal_sparsity_train:.4%}")
print(f"   Test: {temporal_sparsity_test:.4%}")
print("\n✅ Stratégie 1 (Temporal Split) - Terminée")


STRATÉGIE 1: SPLIT TEMPOREL

⚙️  Configuration:
   Train: 80% (les plus anciennes)
   Test: 20% (les plus récentes)

📅 Date de cutoff: 2020-05-09

📊 Split initial:
   Train: 7,995 transactions (80.0%)
   Test: 2,005 transactions (20.1%)

   Train période: 2018-09-20 → 2020-05-08
   Test période: 2020-05-09 → 2020-09-22

🔍 Analyse cold-start:
   Users communs: 34 / 1,996 (1.7%)
   Items communs: 278 / 1,760 (15.8%)

📊 Après filtrage:
   Train: 7,995 transactions
   Test: 2,005 transactions
   Perte: 0 transactions

🔄 Construction des matrices LightFM...
   ✓ Train: (9851, 7865), 7,995 nnz
   ✓ Test: (9851, 7865), 2,005 nnz

📈 Sparsité:
   Train: 99.9897%
   Test: 99.9974%

✅ Stratégie 1 (Temporal Split) - Terminée


In [46]:
print("=" * 80)
print("CONSTRUCTION DES MATRICES DE FEATURES AVEC LIGHTFM")
print("=" * 80)

# ============================================================================
# 5.1 ITEM FEATURES
# ============================================================================
print(f"\n{'='*80}")
print("5.1 ITEM FEATURES")
print(f"{'='*80}")
print(f"\n📋 Features items sélectionnées:")
for col in item_feature_columns:
    n_unique = articles_filtered[col].nunique()
    print(f"   • {col:25s}: {n_unique:3d} catégories")

# Préparer les features au format LightFM : (item_id, [list_of_features])
print(f"\n🔄 Préparation des features au format LightFM...")
item_features_list = []
for idx, row in articles_filtered.iterrows():
    article_id = row['article_id']
    features = []
    for col in item_feature_columns:
        feature_value = row[col]
        features.append(f"{col}:{feature_value}")
    item_features_list.append((article_id, features))
print(f"   ✓ {len(item_features_list):,} items préparés")
print(f"\n   Exemple (3 premiers items):")
for item_id, features in item_features_list[:3]:
    print(f"   • {item_id}: {features[:2]}...")

# Construire la matrice avec LightFM
print(f"\n🔄 Construction de la matrice item_features...")
item_features_matrix = dataset.build_item_features(item_features_list)
print(f"   ✓ Matrice construite")
print(f"\n📊 Caractéristiques de la matrice item features:")
print(f"   Type: {type(item_features_matrix)}")
print(f"   Format: {item_features_matrix.format}")
print(f"   Shape: {item_features_matrix.shape}")
print(f"   Non-zéros (nnz): {item_features_matrix.nnz:,}")
print(f"   Mémoire: {item_features_matrix.data.nbytes / 1024**2:.2f} MB")

# ============================================================================
# 5.2 USER FEATURES
# ============================================================================
print(f"\n{'='*80}")
print("5.2 USER FEATURES")
print(f"{'='*80}")
print(f"\n📋 Features users sélectionnées:")
for col in user_feature_columns:
    n_unique = customers_filtered[col].nunique()
    print(f"   • {col:25s}: {n_unique:2d} catégories")
print(f"\n📊 Distribution des features users:")
print(f"\n   Age groups:")
age_dist = customers_filtered['age_group'].value_counts().sort_index()
for group, count in age_dist.items():
    pct = count / len(customers_filtered) * 100
    print(f"   • {group:8s}: {count:6,} ({pct:5.1f}%)")

# Préparer les features au format LightFM
print(f"\n🔄 Préparation des features au format LightFM...")
user_features_list = []
for idx, row in customers_filtered.iterrows():
    customer_id = row['customer_id']
    features = []
    for col in user_feature_columns:
        feature_value = row[col]
        features.append(f"{col}:{feature_value}")
    user_features_list.append((customer_id, features))
print(f"   ✓ {len(user_features_list):,} users préparés")
print(f"\n   Exemple (3 premiers users):")
for user_id, features in user_features_list[:3]:
    print(f"   • {user_id[:20]}...: {features}")

# Construire la matrice avec LightFM
print(f"\n🔄 Construction de la matrice user_features...")
user_features_matrix = dataset.build_user_features(user_features_list)
print(f"   ✓ Matrice construite")
print(f"\n📊 Caractéristiques de la matrice user features:")
print(f"   Type: {type(user_features_matrix)}")
print(f"   Format: {user_features_matrix.format}")
print(f"   Shape: {user_features_matrix.shape}")
print(f"   Non-zéros (nnz): {user_features_matrix.nnz:,}")
print(f"   Mémoire: {user_features_matrix.data.nbytes / 1024**2:.2f} MB")
print("\n✅ Matrices de features construites avec LightFM")


CONSTRUCTION DES MATRICES DE FEATURES AVEC LIGHTFM

5.1 ITEM FEATURES

📋 Features items sélectionnées:
   • product_group_name       :  13 catégories
   • product_type_name        :  95 catégories
   • garment_group_name       :  21 catégories
   • colour_group_name        :  50 catégories
   • department_name          : 208 catégories

🔄 Préparation des features au format LightFM...
   ✓ 7,865 items préparés

   Exemple (3 premiers items):
   • 108775015: ['product_group_name:Garment Upper body', 'product_type_name:Vest top']...
   • 108775044: ['product_group_name:Garment Upper body', 'product_type_name:Vest top']...
   • 110065001: ['product_group_name:Underwear', 'product_type_name:Bra']...

🔄 Construction de la matrice item_features...
   ✓ Matrice construite

📊 Caractéristiques de la matrice item features:
   Type: <class 'scipy.sparse._csr.csr_matrix'>
   Format: csr
   Shape: (7865, 8252)
   Non-zéros (nnz): 47,190
   Mémoire: 0.18 MB

5.2 USER FEATURES

📋 Features users sélect

---

# Section 8: Modèle Hybride et Analyse

---


## 8.4 Entraînement du Modèle Hybride

### 🎯 Configuration

Nous utilisons les **mêmes hyperparamètres** que le modèle CF pur (Section 6) pour une comparaison équitable.

La seule différence : ajout de `item_features` lors de l'entraînement.


In [51]:
from scipy import sparse
# --- Étape 3 : Prétraitement et Construction de la Matrice ---
print("\n--- Étape 3 : Prétraitement ---")
unique_users = transactions['customer_id'].unique()
unique_items = transactions['article_id'].unique()
user_id_map = {user: i for i, user in enumerate(unique_users)}
item_id_map = {item: i for i, item in enumerate(unique_items)}
user_id_reverse = {v: k for k, v in user_id_map.items()}
item_id_reverse = {v: k for k, v in item_id_map.items()}

rows = transactions['customer_id'].map(user_id_map)
cols = transactions['article_id'].map(item_id_map)
values = np.ones(len(transactions))
interaction_matrix = sparse.csr_matrix((values, (rows, cols)), shape=(len(unique_users), len(unique_items)))
print(f"Matrice d'interactions (sparse) créée : {interaction_matrix.shape}")

# --- Étape 4 : Division Train/Test Temporelle ---
print("\n--- Étape 4 : Division Train/Test ---")
data_sorted = transactions.sort_values('t_dat')
split_point = int(0.8 * len(data_sorted))
train_data = data_sorted[:split_point]
test_data = data_sorted[split_point:]

# === CORRECTION (Binarisation) ===
# S'assurer qu'un couple (user, item) n'apparaît qu'une fois par set
train_data = train_data.drop_duplicates(subset=['customer_id', 'article_id'])
test_data = test_data.drop_duplicates(subset=['customer_id', 'article_id'])
train_data = train_data.drop_duplicates(subset=['customer_id', 'article_id'])
test_data = test_data.drop_duplicates(subset=['customer_id', 'article_id'])
# === FIN CORRECTION ===

# === NOUVELLE CORRECTION : ASSURER LA DISJONCTION TRAIN/TEST ===
# LightFM lève une erreur si des paires (user, item) identiques
# existent dans les deux sets. Nous filtrons le test set.
print("Filtrage des interactions du test déjà vues dans le train...")

# 1. Créer un set de paires uniques (user, item) du train set pour un lookup rapide
train_pairs_set = set(zip(train_data['customer_id'], train_data['article_id']))

# 2. Créer une liste de paires (user, item) du test set
test_pairs_list = list(zip(test_data['customer_id'], test_data['article_id']))

# 3. Créer un masque : True si la paire n'est PAS dans le train set
mask = [pair not in train_pairs_set for pair in test_pairs_list]

# 4. Appliquer le masque pour ne garder que les interactions inconnues
test_data = test_data[mask]

print(f"Interactions du test après filtrage (nouvelles interactions uniques) : {len(test_data)}")
# === FIN NOUVELLE CORRECTION ===

train_rows = train_data['customer_id'].map(user_id_map)
train_cols = train_data['article_id'].map(item_id_map)
train_values = np.ones(len(train_data))
test_rows = test_data['customer_id'].map(user_id_map)
test_cols = test_data['article_id'].map(item_id_map)
test_values = np.ones(len(test_data))

train_matrix = sparse.csr_matrix((train_values, (train_rows, train_cols)), shape=(len(unique_users), len(unique_items)))
test_matrix = sparse.csr_matrix((test_values, (test_rows, test_cols)), shape=(len(unique_users), len(unique_items)))


--- Étape 3 : Prétraitement ---
Matrice d'interactions (sparse) créée : (9851, 7865)

--- Étape 4 : Division Train/Test ---
Filtrage des interactions du test déjà vues dans le train...
Interactions du test après filtrage (nouvelles interactions uniques) : 2000


In [52]:
print(f"   ✓ Train 'train_interactions.npz': {train_matrix.shape} - {train_matrix.nnz:,} interactions")
print(f"   ✓ Test 'test_interactions.npz' : {test_matrix.shape} - {test_matrix.nnz:,} interactions")

   ✓ Train 'train_interactions.npz': (9851, 7865) - 7,999 interactions
   ✓ Test 'test_interactions.npz' : (9851, 7865) - 2,000 interactions


In [65]:
print("=" * 80)
print("ENTRAÎNEMENT MODÈLE HYBRIDE")
print("=" * 80)

# Créer le modèle hybride
hybrid_model = LightFM(
    loss='warp',
    no_components=55,
    learning_rate=0.00394802143257261,
    item_alpha=1.93e-08,
    user_alpha=1.89e-08,
    random_state=42
)

print(f"\n🔄 Entraînement en cours...")
print(f"   Avec item_features ET user_features (LightFM)")

import time
start_time = time.time()

# Entraînement avec les features LightFM
hybrid_model.fit(
    interactions=train_matrix,   # ← Matrices LightFM !
    item_features=item_features_matrix,         # ← Features LightFM (alignées) !
    user_features=user_features_matrix,         # ← Features LightFM (alignées) !
    epochs=10,
    num_threads=4,
    verbose=True
)

training_time = time.time() - start_time
print(f"\n✅ Entraînement terminé en {training_time:.1f}s")

ENTRAÎNEMENT MODÈLE HYBRIDE

🔄 Entraînement en cours...
   Avec item_features ET user_features (LightFM)
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9

✅ Entraînement terminé en 0.2s


In [62]:
print(f"   ✓ Train 'train_interactions.npz': {temporal_train_interactions.shape} - {temporal_train_interactions.nnz:,} interactions")
print(f"   ✓ Test 'test_interactions.npz' : {temporal_test_interactions.shape} - {temporal_test_interactions.nnz:,} interactions")

   ✓ Train 'train_interactions.npz': (9851, 7865) - 7,995 interactions
   ✓ Test 'test_interactions.npz' : (9851, 7865) - 2,005 interactions


## 8.5 Comparaison CF Pur vs Hybrid Model

### 🎯 Objectif

Évaluer si l'ajout de features améliore les performances.

### 📊 Métriques

- Precision@K, Recall@K, AUC sur le test set
- Coverage (diversité du catalogue)


In [66]:
print("=" * 80)
print("ÉVALUATION DU MODÈLE HYBRIDE")
print("=" * 80)

K_VALUES = [5, 10, 20]

print(f"\n🔄 Évaluation du modèle hybride (K={K_VALUES})...")
results_comparison = {'hybrid': {}}

print(f"\n2️⃣  HYBRID MODEL (avec item + user features LightFM):")
print(f"{'K':<6} | {'Precision@K':<13} | {'Recall@K':<13} | {'AUC':<10}")
print("-" * 55)

for k in K_VALUES:
    # Calculer les métriques avec debug
    prec_array = precision_at_k(
        hybrid_model,
        test_matrix,
        k=k,
        train_interactions=train_matrix,
        item_features=item_features_matrix,
        user_features=user_features_matrix,
        num_threads=4
    )

    rec_array = recall_at_k(
        hybrid_model,
        test_matrix,
        k=k,
        train_interactions=train_matrix,
        item_features=item_features_matrix,
        user_features=user_features_matrix,
        num_threads=4
    )

    auc_array = auc_score(
        hybrid_model,
        test_matrix,
        train_interactions=train_matrix,
        item_features=item_features_matrix,
        user_features=user_features_matrix,
        num_threads=4
    )

    # Debug : Afficher statistiques des arrays
    if k == 10:  # Debug uniquement pour K=10
        print(f"\n🔍 Debug Precision@{k}:")
        print(f"   Array shape: {prec_array.shape}")
        print(f"   Users avec P>0: {(prec_array > 0).sum()} / {len(prec_array)} ({(prec_array > 0).sum()/len(prec_array)*100:.1f}%)")
        print(f"   Min/Max: {prec_array.min():.6f} / {prec_array.max():.6f}")

        # Afficher quelques valeurs
        non_zero_idx = np.where(prec_array > 0)[0]
        if len(non_zero_idx) > 0:
            print(f"   Exemples de précisions non-nulles:")
            for idx in non_zero_idx[:5]:
                print(f"      User {idx}: P@{k} = {prec_array[idx]:.4f}")
        else:
            print(f"   ⚠️  AUCUN user avec précision > 0 !")

    prec = prec_array.mean()
    rec = rec_array.mean()
    auc = auc_array.mean()

    results_comparison['hybrid'][k] = {
        'precision': prec,
        'recall': rec,
        'auc': auc
    }
    print(f"{k:<6} | {prec:<13.4f} | {rec:<13.4f} | {auc:<10.4f}")

print(f"\n✅ Évaluation terminée")

# Interprétation
if results_comparison['hybrid'][10]['precision'] == 0.0:
    print(f"\n⚠️  ATTENTION : Precision@10 = 0.0")
    print(f"   Causes possibles:")
    print(f"   1. Test set trop petit ou vide")
    print(f"   2. Items du test non présents dans le train")
    print(f"   3. Modèle sous-entraîné")
    print(f"   4. Dataset trop sparse")
elif results_comparison['hybrid'][10]['precision'] < 0.01:
    print(f"\n⚠️  Performance très faible (P@10 < 1%)")
    print(f"   Considérez d'augmenter la taille du dataset ou les epochs")
else:
    print(f"\n✅ Performance acceptable (P@10 = {results_comparison['hybrid'][10]['precision']:.4f})")

ÉVALUATION DU MODÈLE HYBRIDE

🔄 Évaluation du modèle hybride (K=[5, 10, 20])...

2️⃣  HYBRID MODEL (avec item + user features LightFM):
K      | Precision@K   | Recall@K      | AUC       
-------------------------------------------------------
5      | 0.0002        | 0.0010        | 0.4697    

🔍 Debug Precision@10:
   Array shape: (1991,)
   Users avec P>0: 3 / 1991 (0.2%)
   Min/Max: 0.000000 / 0.100000
   Exemples de précisions non-nulles:
      User 433: P@10 = 0.1000
      User 1458: P@10 = 0.1000
      User 1537: P@10 = 0.1000
10     | 0.0002        | 0.0015        | 0.4697    
20     | 0.0001        | 0.0015        | 0.4697    

✅ Évaluation terminée

⚠️  Performance très faible (P@10 < 1%)
   Considérez d'augmenter la taille du dataset ou les epochs


## 8.6 Analyse Cold-Start

### 🎯 Objectif

Vérifier si le modèle hybride performe mieux sur les **items avec peu d'interactions** (cold-start).

### 📊 Segmentation Items

- **Populaires** : >P75 interactions train
- **Moyens** : P25-P75 interactions
- **Cold-start** : <P25 interactions


In [27]:
print("=" * 80)
print("ANALYSE COLD-START")
print("=" * 80)

# Convertir les matrices en CSR pour permettre l'indexation
train_interactions_csr = train_interactions.tocsr()
test_interactions_csr = test_interactions.tocsr()

# Calculer la popularité des items (nombre d'interactions train)
item_popularity = np.array(train_interactions.sum(axis=0)).flatten()

# Statistiques
print(f"\n📊 Distribution des interactions par item (train):")
print(f"   Min : {item_popularity.min()}")
print(f"   Q25 : {np.percentile(item_popularity, 25):.0f}")
print(f"   Q50 : {np.percentile(item_popularity, 50):.0f}")
print(f"   Q75 : {np.percentile(item_popularity, 75):.0f}")
print(f"   Max : {item_popularity.max()}")

# Définir les seuils
q25 = np.percentile(item_popularity, 25)
q75 = np.percentile(item_popularity, 75)

# Créer les segments d'items
item_segments = {
    'cold_start': np.where(item_popularity < q25)[0],
    'moyens': np.where((item_popularity >= q25) & (item_popularity < q75))[0],
    'populaires': np.where(item_popularity >= q75)[0]
}

print(f"\n📦 Segments d'items créés:")
for seg_name, items in item_segments.items():
    print(f"   • {seg_name.capitalize():<15} : {len(items):>6,} items ({len(items)/num_items*100:>5.1f}%)")

# Pour chaque segment, calculer les métriques
print(f"\n🔄 Évaluation par segment d'items...")

K_COLDSTART = 10

print(f"\n{'Segment':<15} | {'N Items':<10} | {'Hybrid P@10':<15} | {'N Test Interactions':<20}")
print("-" * 70)

coldstart_results = {}

from scipy.sparse import lil_matrix

for seg_name, item_indices in item_segments.items():
    if len(item_indices) == 0:
        continue

    # ============================================================================
    # FIX : Créer une matrice de MÊME TAILLE que train_interactions
    # ============================================================================
    # Créer une matrice vide de la même shape
    test_segment_hybrid = lil_matrix(train_interactions.shape)

    # Copier seulement les colonnes (items) du segment
    for item_idx in item_indices:
        test_segment_hybrid[:, item_idx] = test_interactions_csr[:, item_idx]

    # Convertir en CSR pour efficacité
    test_segment_hybrid = test_segment_hybrid.tocsr()

    # Vérifier qu'il y a des interactions
    if test_segment_hybrid.nnz == 0:
        print(f"{seg_name.capitalize():<15} | {len(item_indices):>10,} | {'N/A':<15} | {0:<20}")
        continue

    # Évaluer Hybrid
    hybrid_prec = precision_at_k(
        hybrid_model,
        test_segment_hybrid,
        k=K_COLDSTART,
        train_interactions=train_interactions,
        item_features=item_features_matrix,
        user_features=user_features_matrix,
        num_threads=4
    ).mean()

    coldstart_results[seg_name] = {
        'n_items': len(item_indices),
        'hybrid_prec': hybrid_prec,
        'n_test_interactions': test_segment_hybrid.nnz
    }

    print(f"{seg_name.capitalize():<15} | {len(item_indices):>10,} | {hybrid_prec:<15.4f} | {test_segment_hybrid.nnz:<20,}")

print(f"\n✅ Analyse cold-start terminée")

print(f"\n💡 INTERPRÉTATION:")
print(f"   Les features aident-elles sur les items cold-start ?")
print(f"   Si Hybrid P@10 sur 'cold_start' > 0, les features généralisent bien.")
print(f"   Si Hybrid P@10 = 0 partout, le dataset est trop sparse pour apprendre.")


ANALYSE COLD-START

📊 Distribution des interactions par item (train):
   Min : 0
   Q25 : 1
   Q50 : 1
   Q75 : 2
   Max : 61

📦 Segments d'items créés:
   • Cold_start      :  3,006 items ( 12.4%)
   • Moyens          : 12,962 items ( 53.5%)
   • Populaires      :  8,248 items ( 34.1%)

🔄 Évaluation par segment d'items...

Segment         | N Items    | Hybrid P@10     | N Test Interactions 
----------------------------------------------------------------------
Cold_start      |      3,006 | 0.0000          | 3,245               
Moyens          |     12,962 | 0.0000          | 2,167               
Populaires      |      8,248 | 0.0011          | 4,586               

✅ Analyse cold-start terminée

💡 INTERPRÉTATION:
   Les features aident-elles sur les items cold-start ?
   Si Hybrid P@10 sur 'cold_start' > 0, les features généralisent bien.
   Si Hybrid P@10 = 0 partout, le dataset est trop sparse pour apprendre.


In [28]:
print(item_popularity)

[2 3 0 ... 1 1 1]
